# Regresi Linear

In [6]:
# Libraries
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_percentage_error , r2_score

In [2]:
# Data Preparation
df = pd.read_excel("../Data Final/Final_PM15-1.xlsx")
df = df[df['MDS'] <= 100]

# Cleaning
df['MDWT'] = pd.to_numeric(df['MDWT'], errors='coerce')
df = df.dropna()

# Cleaning - GSM
df['GSM'] = df['GSM'].astype(str)
df = df[df['GSM'].str.len() <= 4].copy()
df['GSM'] = df['GSM'].str.replace(',', '.')
df['GSM'] = df['GSM'].astype(float)

# Create Pseudo_Mass Feature
df['pseudo_mass'] = (df['Mean_Stock Flow'] * df['Mean_Stock Consistency'])/ df['Mean_Yankee Speed']

# Create Coating-Release Ratio Feature
df['coating_release_ratio'] = df['Mean_Flow Coating'] / df['Mean_Flow Release']

# X Variables
features = [
    'Mean_Creping'
]
X = df[features]


# Y Variables
target = 'MDS'
y = df[target]
stratify_col = df['Grade']

In [3]:
# Split data dengan rasio 80:20 dan stratifikasi
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=stratify_col
)
# Menambahkan intersep (Wajib untuk validasi yang akurat)
X_train_const = sm.add_constant(X_train)
X_test_const = sm.add_constant(X_test)

print("Min MDS :", y.min())
print("Max MDS :", y.max())
print("Min Creping :", X.min())
print("Max Creping :", X.max())
print("MDS Shape :", y.shape)



In [8]:
df.info()

<class 'pandas.DataFrame'>
Index: 439 entries, 21 to 1072
Data columns (total 41 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   Start_Time                      439 non-null    datetime64[us]
 1   End_Time                        439 non-null    datetime64[us]
 2   Data_Count                      439 non-null    int64         
 3   Mean_Creping                    439 non-null    float64       
 4   Mean_Yankee Speed               439 non-null    float64       
 5   Mean_Pope Reel Speed            439 non-null    float64       
 6   Mean_Yankee Pressure            439 non-null    float64       
 7   Mean_Stock Flow                 439 non-null    float64       
 8   Mean_Stock Consistency          439 non-null    float64       
 9   Mean_Flow Coating               439 non-null    float64       
 10  Mean_Flow Release               439 non-null    float64       
 11  Mean_Jet Wire Ratio 

In [4]:
# Model Regresi Linear
model = sm.OLS(y_train, X_train_const).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                    MDS   R-squared:                       0.872
Model:                            OLS   Adj. R-squared:                  0.872
Method:                 Least Squares   F-statistic:                     2385.
Date:                Wed, 20 May 2026   Prob (F-statistic):          4.66e-158
Time:                        13:37:36   Log-Likelihood:                -688.33
No. Observations:                 351   AIC:                             1381.
Df Residuals:                     349   BIC:                             1388.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            1.5102      0.514      2.936   

In [5]:
# Stratified data predict
y_pred = model.predict(X_test_const)

In [7]:
# Evaluasi pada Data Uji (Test Metrics)
mape = mean_absolute_percentage_error(y_test, y_pred) * 100
r2 = r2_score(y_test, y_pred) 

print("\n--- Evaluasi Data Uji ---")
print(f"R-squared: {r2:.4f}")
print(f"MAPE: {mape:.2f}%")


--- Evaluasi Data Uji ---
R-squared: 0.8098
MAPE: 6.30%


In [10]:
# --- OPSI 1: Model Tanpa Intercept (Sesuai kode awal Anda) ---
model_no_const = sm.OLS(y, X).fit()
y_pred_no = model_no_const.predict(X)

mape_no = np.mean(np.abs((y - y_pred_no) / y)) * 100
r2_no = model_no_const.rsquared  # Mengambil R-sq uncentered

print("--- Model Tanpa Intercept ---")
print(f"R-squared: {r2_no:.4f}")
print(f"MAPE: {mape_no:.2f}%")

# --- OPSI 2: Model Dengan Intercept (Disarankan untuk validasi) ---
X_with_const = sm.add_constant(X)
model_with_const = sm.OLS(y, X_with_const).fit()
y_pred_with = model_with_const.predict(X_with_const)

mape_with = np.mean(np.abs((y - y_pred_with) / y)) * 100
r2_with = model_with_const.rsquared # Mengambil R-sq standar

print("\n--- Model Dengan Intercept ---")
print(f"R-squared: {r2_with:.4f}")
print(f"MAPE: {mape_with:.2f}%")

--- Model Tanpa Intercept ---
R-squared: 0.9953
MAPE: 5.54%

--- Model Dengan Intercept ---
R-squared: 0.8599
MAPE: 5.49%


In [11]:
print(model_with_const.summary())

                            OLS Regression Results                            
Dep. Variable:                    MDS   R-squared:                       0.860
Model:                            OLS   Adj. R-squared:                  0.860
Method:                 Least Squares   F-statistic:                     2683.
Date:                Wed, 20 May 2026   Prob (F-statistic):          1.24e-188
Time:                        10:42:17   Log-Likelihood:                -881.22
No. Observations:                 439   AIC:                             1766.
Df Residuals:                     437   BIC:                             1775.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const            1.6922      0.482      3.510   

In [12]:
print(model_no_const.summary())

                                 OLS Regression Results                                
Dep. Variable:                    MDS   R-squared (uncentered):                   0.995
Model:                            OLS   Adj. R-squared (uncentered):              0.995
Method:                 Least Squares   F-statistic:                          9.317e+04
Date:                Wed, 20 May 2026   Prob (F-statistic):                        0.00
Time:                        10:42:17   Log-Likelihood:                         -887.32
No. Observations:                 439   AIC:                                      1777.
Df Residuals:                     438   BIC:                                      1781.
Df Model:                           1                                                  
Covariance Type:            nonrobust                                                  
                   coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------

### Import Model

In [ ]:
import joblib
import os

# Folder model
# os.makedirs("models/PM15/MDS", exist_ok=True)

# Simpan model
# joblib.dump(model_no_const, "models/PM15/MDS/model.pkl")

# Simpan fitur
# joblib.dump(features, "models/PM15/MDS/features.pkl")

print("Model berhasil disimpan!")

Model berhasil disimpan!


## Pemeriksaan Asumsi IIDN

### Independen (Autokorelasi)

In [13]:
DW = sm.stats.durbin_watson(model.resid)
print('Nilai DW =', DW)
if DW > 1.5 and DW < 2.5:
    print('Tidak ada autokorelasi (baik positif maupun negatif)')
elif DW < 1.5:
    print('Terdapat autokorelasi positif.')
elif DW > 2.5:
    print('Terdapat autokorelasi negatif.')

Nilai DW = 0.9290706928710329
Terdapat autokorelasi positif.


### Indentik (Heterokesdastisitas)

In [14]:
from statsmodels.stats.diagnostic import het_breuschpagan

# Breusch-Pagan Test
# H0: Homoskedastisitas (Varians konstan)
bp_test = het_breuschpagan(model.resid, model.model.exog)
labels = ['LM Statistic', 'LM-Test p-value', 'F-Statistic', 'F-Test p-value']
print(dict(zip(labels, bp_test)))

ValueError: The Breusch-Pagan test requires exog to have at least two columns where one is a constant.

Terjadi Heteroskedastisitas yang parah pada model. Varians dari residual terbukti secara empiris tidak konstan.

### Normalitas

In [ ]:
# Libraries
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
# Histogram
plt.figure()
plt.hist(model.resid, bins=30)
plt.title("Histogram Residual")
plt.show()

In [ ]:
# Visualisasi (Q-Q Plot)
sm.qqplot(model.resid, line='45', fit=True)
plt.title("Q-Q Plot Residual")
plt.show()

In [ ]:
# Uji Statistik (Shapiro-Wilk)
# H0: Data berdistribusi normal
# H1: Data tidak berdistribusi normal

stat, p_value = stats.shapiro(model.resid)

# Jika p-value < 0.05, residual tidak berdistribusi normal
print(f"Shapiro-Wilk Test: p-value = {p_value}")
if p_value < 0.05:
    print('residual tidak berdistribusi normal')
else:
    print('residual berdistribusi normal.')

### Multikolinearitas

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Menghitung VIF untuk setiap variabel independen
vif_data = pd.DataFrame()
vif_data["Feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]

print(vif_data)